# CrossWordBench Eval — Qwen3-4B baseline

Serve **Qwen3-4B** with vLLM and run the CrossWordBench-derived eval in the same session (against `localhost:8000` — **no tunnel**). Each puzzle's word set + size + black-square count become a SPEC; the model produces a crossword configuration; we score **success** (valid config) plus **crossings**, **coverage**, and **black-square delta**.

**Runtime:** GPU (T4 is enough for a 4B model). **Order:** run top to bottom.

## 1. Get the code
Clone the repo (set your URL) or upload the project folder and set `PROJECT_DIR`. The `bench/`, `harness/`, `pipeline/`, and `seeds/` dirs must be present.

In [ ]:
REPO_URL = "https://github.com/Avaneesh-Ramesh-07/CrosswordSLM.git"
import os
!git clone -q $REPO_URL slm || echo "clone skipped/failed — upload the folder instead"
PROJECT_DIR = "/content/slm"   # adjust if you uploaded elsewhere
assert os.path.isdir(os.path.join(PROJECT_DIR, "bench")), "Set PROJECT_DIR to the repo root"
%cd $PROJECT_DIR

## 2. Install dependencies
The eval harness is pure-Python stdlib; we only need vLLM (to serve the model) and requests (to poll it).

In [ ]:
!pip -q install vllm requests

## 3. Pull the CrossWordBench data (gated)
The dataset is gated, so you need a HuggingFace **read** token whose account has **accepted the terms** at huggingface.co/datasets/HINT-lab/CrossWordBench. The token is read via getpass (not stored in the notebook). Pulls the `english` config (7x7 + 14x14) as compact JSONL into `data/crosswordbench/`.

In [ ]:
import os, getpass
os.environ["HF_TOKEN"] = getpass.getpass("HF read token (gated access accepted): ")
!python bench/pull_crosswordbench.py --config english
!ls -la data/crosswordbench/

## 4. Serve Qwen3-4B via vLLM
T4: keep `--dtype half`. A100/L4: `bfloat16` and a larger `--max-model-len` are fine.

In [ ]:
import subprocess, sys, time, requests

MODEL = "Qwen/Qwen3-4B"
LOG = "vllm.log"
server = subprocess.Popen(
    [sys.executable, "-m", "vllm.entrypoints.openai.api_server",
     "--model", MODEL, "--dtype", "half", "--max-model-len", "8192",
     "--gpu-memory-utilization", "0.90", "--port", "8000"],
    stdout=open(LOG, "w"), stderr=subprocess.STDOUT)
print(f"launched vLLM pid={server.pid} for {MODEL}; first run downloads ~8GB…", flush=True)

def _tail(n=8):
    try:
        with open(LOG) as fh: return "".join(fh.readlines()[-n:]).rstrip()
    except FileNotFoundError: return "(no log yet)"

start, DEADLINE, up, i = time.time(), 1500, False, 0
while time.time() - start < DEADLINE:
    el = int(time.time() - start)
    rc = server.poll()
    if rc is not None:                       # crashed -> stop now, show why
        print(f"\n[{el}s] vLLM EXITED rc={rc}. Last log:\n", flush=True)
        print(_tail(40), flush=True); break
    try:
        if requests.get("http://localhost:8000/v1/models", timeout=2).ok:
            up = True; print(f"\n[{el}s] vLLM UP — {MODEL} ready on :8000", flush=True); break
    except Exception:
        pass
    if i % 3 == 0:                           # heartbeat + latest log line ~every 15s
        print(f"[{el:4d}s] loading… | {_tail(1)}", flush=True)
    i += 1; time.sleep(5)
if not up and server.poll() is None:
    print(f"\n[{int(time.time()-start)}s] TIMEOUT — still not serving.", flush=True)
if not up:
    print("\n===== tail vllm.log =====\n" + _tail(40), flush=True)

## 5. Baselines / anchors (no model needed)
`reference` scores each puzzle's OWN grid. Two validity modes:
- **strict** (NYT rules: every cell checked both ways + symmetry) — references score ~0 (they're loose auto-fills), so this is a hard target.
- **relaxed** (`--relaxed`, CrossWordBench-style: unchecked cells allowed, but every cell in a real entry, no conflicts, connected) — references score ~1.0, so `success` is meaningful.

`seed:reference_v1` runs a hand-written CSP generator as a 'model' — a floor showing how hard a valid fill is from only the ~12 given words.

In [ ]:
!python bench/run_crosswordbench.py --model reference --config english            # strict
!python bench/run_crosswordbench.py --model reference --config english --relaxed  # relaxed (≈1.0)
!python bench/run_crosswordbench.py --model seed:reference_v1 --config english --limit 20 --relaxed

## 6. Smoke test: Qwen3-4B on 20 puzzles
Confirms the endpoint round-trip and response parsing before the full run. `--mode program` = model writes a `generate_crossword` program we run in the sandbox (matches your trained SLM's interface). Use `--mode direct` to have it emit the layout JSON itself.

In [ ]:
!python bench/run_crosswordbench.py --model endpoint --mode program \
    --base-url http://localhost:8000/v1 --model-name Qwen/Qwen3-4B \
    --config english --split 7x7 --limit 20

## 7. Full eval + save per-puzzle results
Runs all 200 english puzzles. `--mode program` (matches your trained SLM's interface) is reported under both **strict** and **relaxed** validity; `--mode direct` (model emits the layout JSON) under relaxed. Per-puzzle rows go to `runs/eval/*.jsonl` for later analysis and for comparison against the tuned SLM. Success may be ~0 under strict — the crossings / coverage / black-delta columns are where the signal is.

In [ ]:
import os
os.makedirs("runs/eval", exist_ok=True)
EP = "--base-url http://localhost:8000/v1 --model-name Qwen/Qwen3-4B --config english"
!python bench/run_crosswordbench.py --model endpoint --mode program $EP \
    --out runs/eval/qwen3_4b_program_strict.jsonl
!python bench/run_crosswordbench.py --model endpoint --mode program $EP --relaxed \
    --out runs/eval/qwen3_4b_program_relaxed.jsonl
!python bench/run_crosswordbench.py --model endpoint --mode direct $EP --relaxed \
    --out runs/eval/qwen3_4b_direct_relaxed.jsonl

## 8. Save results to Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
!mkdir -p /content/drive/MyDrive/slm_runs/eval
!cp -r runs/eval/* /content/drive/MyDrive/slm_runs/eval/ 2>/dev/null || true
print("saved runs/eval to Drive")

## Next
These are the **base-model** numbers. After QLoRA training, re-run cell 7 pointing `--model-name` at the tuned checkpoint (served the same way) to get the base-vs-tuned delta on identical held-out puzzles.